## VIT Setup

In [ ]:
# imports
import os
import sqlite3
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys
import itertools
import math
import gdown
from torch.cuda.amp import autocast, GradScaler

print(f"PyTorch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
DB_PATH = './beatmaps.db'

try:
  import google.colab  # type: ignore
  if not os.path.exists('/content/beatmaps.db'):
    url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
    output = '/content/beatmaps.db'
    gdown.download(url, output, quiet=False)
  DB_PATH = '/content/beatmaps.db'
except Exception:
    pass

print(f"Database path: {DB_PATH}")

MAX_SEQ_LEN = 1023
IN_CHANNELS = 10

D_MODEL = 256
N_HEADS = 8
N_LAYERS = 6
DIM_FEEDFORWARD = 4 * D_MODEL
DROPOUT = 0.1

BATCH_SIZE = 8

In [ ]:
def load_and_group_data_from_db_fast_optimized(db_path, chunk_size=1000):
    print("Connecting to database...")
    con = sqlite3.connect(db_path)
    cursor = con.cursor()

    print("Fetching valid beatmap IDs and all metadata...")
    metadata_df = pd.read_sql_query(
        """
        SELECT id, ar, od, circle_size as cs, difficulty_rating, main_bpm 
        FROM beatmaps 
        WHERE main_bpm IS NOT NULL 
        ORDER BY id
        """, 
        con
    )
    valid_map_ids = metadata_df['id'].tolist()
    
    metadata_dict = {
        row.id: np.array([row.ar, row.od, row.cs, row.difficulty_rating, row.main_bpm], dtype=np.float32)
        for row in metadata_df.itertuples(index=False)
    }
    
    print(f"Found {len(valid_map_ids)} beatmaps with complete metadata.")

    processed_data = []
    num_chunks = (len(valid_map_ids) + chunk_size - 1) // chunk_size

    vector_query_template = """
        SELECT beatmap_id, x_diff, y_diff, time_diff, abs_x, abs_y, object_type, is_new_combo, slider_curve_type, slider_num_anchors, slider_pixel_length
        FROM beatmap_vectors 
        WHERE beatmap_id IN ({placeholders}) 
        ORDER BY beatmap_id
    """

    for i in tqdm(
        range(0, len(valid_map_ids), chunk_size),
        total=num_chunks,
        desc="Processing Chunks",
        dynamic_ncols=True,
        leave=True,
        file=sys.stdout
    ):
        chunk_ids = valid_map_ids[i:i + chunk_size]
        
        placeholders = ','.join('?' for _ in chunk_ids)
        query = vector_query_template.format(placeholders=placeholders)
        
        cursor.execute(query, chunk_ids)
        
        for map_pk, group_iter in itertools.groupby(cursor, key=lambda row: row[0]):
            
            vectors_list = [row[1:] for row in group_iter]
            
            if not vectors_list:
                continue

            meta_np = metadata_dict.get(map_pk)
            if meta_np is None:
                continue 

            vectors_tensor = torch.tensor(vectors_list, dtype=torch.float32)
            vectors_tensor[:, 2] = torch.log1p(vectors_tensor[:, 2])
            metadata_tensor = torch.from_numpy(meta_np) 
            
            processed_data.append((vectors_tensor, metadata_tensor))

    con.close()
    print("Finished processing all data.")
    return processed_data

all_beatmaps_data = load_and_group_data_from_db_fast_optimized(DB_PATH, chunk_size=1000)

METADATA_DIM = all_beatmaps_data[0][1].shape[0]
print(f"\nDetected metadata dimension: {METADATA_DIM}")

In [ ]:
from torch.utils.data import random_split, WeightedRandomSampler

print("Calculating weights for sampling based on difficulty rating...")
all_difficulty_ratings = np.array([data[1][3].item() for data in all_beatmaps_data])

bins = [0, 5, 6, 7, 8, 9, 10, np.inf]
binned_ratings = pd.cut(all_difficulty_ratings, bins=bins, right=False, labels=False)
class_counts = np.bincount(binned_ratings, minlength=len(bins)-1)

print("\n--- Difficulty Distribution ---")
for i in range(len(bins)-1):
    lower = bins[i]
    upper = bins[i+1]
    count = class_counts[i]
    total_maps = len(all_beatmaps_data)
    percentage = (count / total_maps * 100) if total_maps > 0 else 0
    if np.isinf(upper):
        print(f"{lower:2.0f}★+  : {count:7d} maps ({percentage:5.2f}%)")
    else:
        print(f"{lower:2.0f}-{upper:2.0f}★ : {count:7d} maps ({percentage:5.2f}%)")
print("-" * 35)

class_weights = 1.0 / (class_counts + 1e-8)
sample_weights = class_weights[binned_ratings]
sample_weights = torch.from_numpy(sample_weights).double()

val_size = int(len(all_beatmaps_data) * 0.1)
train_size = len(all_beatmaps_data) - val_size
train_data, val_data = random_split(all_beatmaps_data, [train_size, val_size])

train_indices = train_data.indices
train_weights = sample_weights[train_indices]
sampler = WeightedRandomSampler(weights=train_weights, num_samples=len(train_weights), replacement=True)

print(f"Data split into {len(train_data)} training samples and {len(val_data)} validation samples.")
print("WeightedRandomSampler created for the training set to address difficulty imbalance.")


print("\nCalculating normalization statistics from the augmented training set...")

all_vectors_list = [data[0] for data in train_data]
all_metadata_list = [data[1] for data in train_data]

augmented_vectors_list_for_stats = []
for vectors in all_vectors_list:
    augmented_vectors_list_for_stats.append(vectors)
    
    flipped_x = vectors.clone()
    flipped_x[:, 0] *= -1                  
    flipped_x[:, 3] = 512 - flipped_x[:, 3]  
    augmented_vectors_list_for_stats.append(flipped_x)
    
    flipped_y = vectors.clone()
    flipped_y[:, 1] *= -1                  
    flipped_y[:, 4] = 384 - flipped_y[:, 4]  
    augmented_vectors_list_for_stats.append(flipped_y)

    flipped_xy = vectors.clone()
    flipped_xy[:, 0] *= -1                  
    flipped_xy[:, 1] *= -1                  
    flipped_xy[:, 3] = 512 - flipped_xy[:, 3]  
    flipped_xy[:, 4] = 384 - flipped_xy[:, 4]  
    augmented_vectors_list_for_stats.append(flipped_xy)

all_vectors_tensor = torch.cat(augmented_vectors_list_for_stats, dim=0)
all_metadata_tensor = torch.stack(all_metadata_list, dim=0) 

vector_mean = all_vectors_tensor.mean(dim=0)
vector_std = all_vectors_tensor.std(dim=0)

meta_mean = all_metadata_tensor.mean(dim=0)
meta_std = all_metadata_tensor.std(dim=0)

vector_std[vector_std == 0] = 1.0
meta_std[meta_std == 0] = 1.0

print("\n--- Normalization Stats (from augmented training data) ---")
print(f"Vector Mean: {vector_mean.numpy()}")
print(f"Vector Std:  {vector_std.numpy()}")
print(f"Meta Mean:   {meta_mean.numpy()}")
print(f"Meta Std:    {meta_std.numpy()}")

In [ ]:
def collate_fn(batch, max_seq_len, vector_dim):
    """
    Collates a batch of variable-length sequences into padded tensors.
    Also creates an attention mask.
    """
    vectors, metadata = zip(*batch)
    
    lengths = [min(v.shape[0], max_seq_len) for v in vectors]
    max_len_batch = max(lengths) if lengths else 0
    
    padded_vectors = torch.zeros(len(batch), max_len_batch, vector_dim, dtype=torch.float32)
    attention_mask = torch.zeros(len(batch), max_len_batch, dtype=torch.bool)
    
    for i, (v, length) in enumerate(zip(vectors, lengths)):
        if length > 0:
            padded_vectors[i, :length] = v[:length]
            attention_mask[i, :length] = True
        
    stacked_metadata = torch.stack(metadata, dim=0)
    
    return padded_vectors.to(device), attention_mask.to(device), stacked_metadata.to(device)


class BeatmapDataset(Dataset):
    """
    PyTorch Dataset for osu! beatmaps.
    Handles data loading, on-the-fly augmentation, and normalization.
    Augmentation is now randomized within __getitem__ to be compatible with WeightedRandomSampler.
    """
    def __init__(self, beatmap_data, vector_mean, vector_std, meta_mean, meta_std, augment=False):
        self.beatmap_data = beatmap_data
        self.vector_mean = vector_mean
        self.vector_std = vector_std
        self.meta_mean = meta_mean
        self.meta_std = meta_std
        self.epsilon = 1e-8
        self.augment = augment

    def __len__(self):
        return len(self.beatmap_data)

    def __getitem__(self, idx):
        vectors, metadata = self.beatmap_data[idx]

        vectors = vectors.clone()
        if self.augment:
            aug_type = torch.randint(0, 4, (1,)).item()
            if aug_type == 1: # Flip X
                vectors[:, 0] *= -1                  
                vectors[:, 3] = 512 - vectors[:, 3]  
            elif aug_type == 2: # Flip Y
                vectors[:, 1] *= -1                  
                vectors[:, 4] = 384 - vectors[:, 4]  
            elif aug_type == 3: # Flip XY
                vectors[:, 0:2] *= -1                
                vectors[:, 3] = 512 - vectors[:, 3]  
                vectors[:, 4] = 384 - vectors[:, 4]  

        normalized_metadata = (metadata - self.meta_mean) / (self.meta_std + self.epsilon)
        normalized_vectors = (vectors - self.vector_mean) / (self.vector_std + self.epsilon)

        return normalized_vectors, normalized_metadata

train_dataset = BeatmapDataset(train_data, vector_mean, vector_std, meta_mean, meta_std, augment=True)
val_dataset = BeatmapDataset(val_data, vector_mean, vector_std, meta_mean, meta_std, augment=False)

collate_with_args = lambda batch: collate_fn(batch, max_seq_len=MAX_SEQ_LEN, vector_dim=IN_CHANNELS)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, collate_fn=collate_with_args)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_with_args)

print(f"Created a training dataset with {len(train_dataset)} unique maps (randomly augmented per fetch).")
print(f"Created a validation dataset with {len(val_dataset)} maps.")
print("\nA sample batch will be a tuple: (vectors, attention_mask, metadata)")
sample_vecs, sample_mask, sample_meta = next(iter(train_dataloader))
print(f"Vectors shape: {sample_vecs.shape}, Mask shape: {sample_mask.shape}, Meta shape: {sample_meta.shape}")

In [ ]:
import torch.nn.functional as F

def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    """Precomputes the complex numbers for RoPE rotations."""
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs) 
    return freqs_cis

def apply_rotary_emb(xq, xk, freqs_cis):
    """Applies RoPE to query and key tensors."""
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2) 
    
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

class MultiHeadAttentionWithRoPE(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout
        
    def forward(self, x, freqs_cis, mask):
        batch_size, seq_len, _ = x.shape
        
        q, k, v = self.wq(x), self.wk(x), self.wv(x)
        
        q = q.view(batch_size, seq_len, self.n_heads, self.d_head)
        k = k.view(batch_size, seq_len, self.n_heads, self.d_head)
        v = v.view(batch_size, seq_len, self.n_heads, self.d_head)

        q, k = apply_rotary_emb(q, k, freqs_cis)
        
        q = q.transpose(1, 2)
        k = k.transpose(1, 2) 
        v = v.transpose(1, 2) 

        attn_mask = mask.unsqueeze(1).unsqueeze(2) 
        attn_mask = attn_mask == False 

        output = F.scaled_dot_product_attention(
            q, k, v, 
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0
        )
        
        # 6. Reshape and project output
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.wo(output)

class TransformerEncoderLayerWithRoPE(nn.Module):
    def __init__(self, d_model, n_heads, dim_feedforward, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttentionWithRoPE(d_model, n_heads, dropout)
        self.w1 = nn.Linear(d_model, dim_feedforward)
        self.w2 = nn.Linear(dim_feedforward, d_model)
        self.w3 = nn.Linear(d_model, dim_feedforward)
        self.norm1 = nn.RMSNorm(d_model)
        self.norm2 = nn.RMSNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, freqs_cis, src_key_padding_mask):
        src2 = self.self_attn(self.norm1(src), freqs_cis, src_key_padding_mask)
        src = src + self.dropout1(src2)
        
        normalized_src = self.norm2(src)
        ffn_output = self.w2(F.silu(self.w1(normalized_src)) * self.w3(normalized_src))
        src = src + self.dropout2(ffn_output)
        
        return src

class OsuBert(nn.Module):
    def __init__(self, *, max_seq_len, d_model, n_heads, n_layers, dim_feedforward, dropout, metadata_dim, in_channels):
        super().__init__()
        self.d_model = d_model
        
        self.input_proj = nn.Linear(in_channels, d_model)
        self.metadata_proj = nn.Linear(metadata_dim, d_model)
        self.metadata_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.layers = nn.ModuleList([
            TransformerEncoderLayerWithRoPE(d_model, n_heads, dim_feedforward, dropout)
            for _ in range(n_layers)
        ])
        
        self.register_buffer("freqs_cis", precompute_freqs_cis(d_model // n_heads, max_seq_len + 1))

    def encode(self, embeddings, padding_mask):
        """Runs the transformer encoder layers on already-embedded inputs."""
        freqs_cis_slice = self.freqs_cis[:embeddings.shape[1]]
        
        output = embeddings
        for layer in self.layers:
            output = layer(output, freqs_cis=freqs_cis_slice, src_key_padding_mask=padding_mask)
        return output

    def forward(self, x, metadata, attention_mask):
        """Performs full forward pass from raw inputs to final embeddings."""
        batch_size, _, _ = x.shape
        
        # --- Embedding Stage ---
        x_embed = self.input_proj(x)
        meta_embed = self.metadata_proj(metadata).unsqueeze(1) + self.metadata_token
        full_embeddings = torch.cat([meta_embed, x_embed], dim=1)
        
        # --- Mask Creation ---
        meta_mask = torch.zeros((batch_size, 1), dtype=torch.bool, device=x.device)
        padding_mask = ~attention_mask
        full_padding_mask = torch.cat([meta_mask, padding_mask], dim=1)
        
        # --- Encoding Stage ---
        output = self.encode(full_embeddings, full_padding_mask)
            
        return output

In [ ]:
sample_vectors, sample_mask, sample_metadata = next(iter(train_dataloader))

model = OsuBert(
    max_seq_len=MAX_SEQ_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    in_channels=IN_CHANNELS,
    metadata_dim=METADATA_DIM
).to(device)

with torch.no_grad():
    output = model(sample_vectors, sample_metadata, sample_mask)

print("--- Model Sanity Check ---")
print(f"Input vectors shape:  {sample_vectors.shape}")
print(f"Input mask shape:     {sample_mask.shape}")
print(f"Input metadata shape: {sample_metadata.shape}")
print(f"Output tensor shape:  {output.shape}")

actual_batch_size, actual_seq_len = sample_vectors.shape[0], sample_vectors.shape[1]
expected_shape = (actual_batch_size, actual_seq_len + 1, D_MODEL)
print(f"Expected output shape: {expected_shape}")

assert output.shape == expected_shape, "Mismatch between output and expected shape!"
print("\n✅ Sanity check passed! The BERT-style architecture is working correctly.")

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {num_params / 1e6:.2f}M")

## MLM Training

In [ ]:
class OsuBertForMaskedModeling(nn.Module):
    def __init__(self, bert_model, in_channels, masking_ratio=0.15):
        super().__init__()
        self.bert = bert_model
        self.masking_ratio = masking_ratio
        
        self.mask_token_embed = nn.Parameter(torch.randn(1, 1, bert_model.d_model))
        
        self.prediction_head = nn.Sequential(
            nn.Linear(bert_model.d_model, bert_model.d_model),
            nn.GELU(),
            nn.LayerNorm(bert_model.d_model),
            nn.Linear(bert_model.d_model, in_channels)
        )

    def forward(self, x, metadata, attention_mask):
        x_embed = self.bert.input_proj(x)
        meta_embed = self.bert.metadata_proj(metadata).unsqueeze(1) + self.bert.metadata_token
        
        prob = torch.full(x_embed.shape[:2], self.masking_ratio, device=x.device)
        prob.masked_fill_(~attention_mask, 0.0)
        is_masked = torch.bernoulli(prob).bool()

        if not is_masked.any():
            return torch.tensor([], device=x.device), torch.tensor([], device=x.device)

        targets = x[is_masked]

        mask_expanded = is_masked.unsqueeze(-1).expand_as(x_embed)
        encoder_x_input = torch.where(mask_expanded, self.mask_token_embed, x_embed)
        
        full_encoder_input = torch.cat([meta_embed, encoder_x_input], dim=1)

        batch_size = x.shape[0]
        meta_pad_mask = torch.zeros((batch_size, 1), dtype=torch.bool, device=x.device)
        seq_pad_mask = ~attention_mask
        full_padding_mask = torch.cat([meta_pad_mask, seq_pad_mask], dim=1)
        
        encoded_output = self.bert.encode(full_encoder_input, full_padding_mask)
        
        encoded_masked_tokens = encoded_output[:, 1:, :][is_masked]
        predictions = self.prediction_head(encoded_masked_tokens)
        
        return predictions, targets


bert_encoder_rope = OsuBert(
    max_seq_len=MAX_SEQ_LEN, d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD, dropout=DROPOUT, in_channels=IN_CHANNELS, metadata_dim=METADATA_DIM
)

mlm_model = OsuBertForMaskedModeling(bert_encoder_rope, IN_CHANNELS).to(device)

sample_vectors, sample_mask, sample_metadata = next(iter(train_dataloader))

pred, orig = mlm_model(sample_vectors, sample_metadata, sample_mask)

print("--- MLM Model (with RoPE) Sanity Check ---")
print(f"Input vectors shape: {sample_vectors.shape}")
print(f"Predicted masked vectors shape: {pred.shape}")
print(f"Original masked vectors shape:  {orig.shape}")

assert pred.shape[0] == orig.shape[0] and pred.shape[0] > 0, "Shape mismatch or no tokens were masked!"
assert pred.shape[1] == IN_CHANNELS and orig.shape[1] == IN_CHANNELS, "Vector dimension mismatch!"
print("\n✅ Sanity check passed! MLM I/O shapes are correct with RoPE implementation.")

num_params_mlm = sum(p.numel() for p in mlm_model.parameters() if p.requires_grad)
print(f"\nTotal trainable MLM parameters: {num_params_mlm / 1e6:.2f}M")

In [ ]:
import torch
import torch.nn as nn
import time
import math
from tqdm.auto import tqdm

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.05
NUM_EPOCHS = 5
total_steps = len(train_dataloader) * NUM_EPOCHS
WARMUP_RATIO = 0.05
warmup_steps = int(WARMUP_RATIO * total_steps)
MIN_LR = 1e-6
base_lr = LEARNING_RATE
eta_min = MIN_LR
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = (device == 'cuda')

CHECKPOINT_DIR = "./checkpoints"
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "mlm_bert_rope_latest.pth")
os.makedirs(CHECKPOINT_DIR, exist_ok=True) 

bert_encoder_rope = OsuBert(
    max_seq_len=MAX_SEQ_LEN, d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD, dropout=DROPOUT, in_channels=IN_CHANNELS, metadata_dim=METADATA_DIM
)
mlm_model = OsuBertForMaskedModeling(bert_encoder_rope, IN_CHANNELS, masking_ratio=0.25).to(device)

optimizer = torch.optim.AdamW(mlm_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()
use_amp = (device == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# mlm_model = torch.compile(mlm_model, mode="reduce-overhead")

def lr_lambda(current_step: int):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = (current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
    return (eta_min / base_lr) + (1.0 - (eta_min / base_lr)) * cosine_decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    print(f"--- Found checkpoint at {CHECKPOINT_PATH}. Loading... ---")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    
    mlm_model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    
    print(f"--- Resuming training from Epoch {start_epoch + 1} ---")
else:
    print("--- No checkpoint found. Starting training from scratch. ---")

print("\n--- Starting BERT Pre-training (MLM with RoPE) ---")
print(f"Device: {device}, AMP Enabled: {use_amp}")
print(f"Total Epochs: {NUM_EPOCHS}, Starting from Epoch: {start_epoch + 1}")
print(f"Batch Size: {BATCH_SIZE}, Base LR: {LEARNING_RATE}")
print(f"Total training steps: {total_steps}, Warmup steps: {warmup_steps}")
print("-" * 40)

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start_time = time.time()
    
    mlm_model.train()
    train_loss = 0.0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", dynamic_ncols=True)
    
    for vectors, attention_mask, metadata in progress_bar:
        optimizer.zero_grad()
        
        with torch.amp.autocast(device_type=device, dtype=torch.bfloat16, enabled=use_amp):
            predicted_vectors, original_vectors = mlm_model(vectors, metadata, attention_mask)
            if predicted_vectors.numel() > 0:
                loss = loss_fn(predicted_vectors, original_vectors)
            else:
                loss = torch.tensor(0.0, device=device) 
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        train_loss += loss.item()
        progress_bar.set_postfix({
            "Loss": f"{loss.item():.4f}",
            "LR": f"{optimizer.param_groups[0]['lr']:.2e}"
        })
        
    avg_train_loss = train_loss / len(train_dataloader)

    mlm_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for vectors, attention_mask, metadata in val_dataloader:
            with torch.amp.autocast(device_type=device, dtype=torch.bfloat16, enabled=use_amp):
                predicted_vectors, original_vectors = mlm_model(vectors, metadata, attention_mask)
                if predicted_vectors.numel() > 0:
                    loss = loss_fn(predicted_vectors, original_vectors)
                    val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_dataloader) if len(val_dataloader) > 0 else 0.0
    
    epoch_duration = time.time() - epoch_start_time
    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
        f"Time: {epoch_duration:.2f}s"
    )

    checkpoint_data = {
        'epoch': epoch,
        'model_state_dict': mlm_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'val_loss': avg_val_loss,
    }
    torch.save(checkpoint_data, CHECKPOINT_PATH)
    print(f"Checkpoint saved for epoch {epoch+1} to {CHECKPOINT_PATH}")
    print("-" * 40)


print("\n--- Training Finished ---")